# SCI1070 — Semaine 6 : quatre réparations, et laquelle sert vraiment

| Volet | Ce que vous y faites | Statut |
| --- | --- | --- |
| A | lire la bibliothèque, et **prédire sa sortie avant de l'exécuter** | 10 minutes |
| B | le socle commun, et la courbe de référence sur CartPole | implémentation |
| C | **l'ablation à quatre variantes, trois graines** | **évalué** |
| D | mille états témoins, et la contrainte de moyenne nulle | méthode |
| E | couches bruitées et tampon priorisé | facultatif |

Le **volet C est, littéralement, la moitié du jalon 2** de la semaine prochaine.

> **La mise en garde la plus importante du laboratoire.** Je ne vous donne, dans
> aucun volet, le nombre de trames que vous devez obtenir. Tous les chiffres de
> cette semaine viennent d'exécutions sur Atari avec des hyperparamètres réglés
> variante par variante. Les attentes sont **qualitatives** : un ordre entre les
> courbes, un signe, une forme. **Si l'ordre de vos courbes est stable sur trois
> graines, vous avez un résultat.**

In [ ]:
import importlib.util
import subprocess
import sys

for paquet, module in [("gymnasium[classic-control]>=1.0", "gymnasium"), ("torch", "torch")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)

import collections
import math
from dataclasses import dataclass
from typing import Optional

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

peripherique = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("périphérique :", peripherique)

---
# Volet A : lire la bibliothèque, et prédire sa sortie

**Objectif.** Vérifier que vous avez compris la transition premier-dernier, **sans
entraîner quoi que ce soit**. C'est la **seule** semaine du cours où une bibliothèque
d'agents est un objet d'étude plutôt qu'une solution courte.

In [ ]:
# Deux lignes, et les deux précautions comptent (section 6.1). PTAN 0.8 fige
# torch==2.3.1, qui n'a plus de roue pour les Python récents : sans `--no-deps`,
# l'installation s'arrête avant d'avoir rien installé. Et son premier import charge
# stable_baselines3, qu'elle ne déclare pas dans ses dépendances.
ptan_disponible = False
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "stable_baselines3"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "ptan==0.8", "--no-deps"], check=True)
    import ptan.experience as ptan_experience
    ptan_disponible = True
    print("ptan chargé, ExperienceSourceFirstLast :",
          hasattr(ptan_experience, "ExperienceSourceFirstLast"))
except Exception as exc:
    print("ptan indisponible ici :", type(exc).__name__, exc)

### L'environnement jouet de la section 6.3

Une observation qui compte de 0 à 4, une récompense **égale à l'action jouée**, un
épisode de dix pas exactement, et un agent qui joue toujours l'action 1.

In [ ]:
class JouetCompteur(gym.Env):
    """Observation 0..4, récompense = action, épisode de dix pas."""

    def __init__(self):
        self.observation_space = gym.spaces.Discrete(5)
        self.action_space = gym.spaces.Discrete(3)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.pas = 0
        return 0, {}

    def step(self, action):
        self.pas += 1
        obs = self.pas % 5
        return obs, float(action), False, self.pas >= 10, {}


@dataclass
class TransitionPremierDernier:
    """Ce que consomme la perte d'un DQN, à un pas comme à n étapes."""
    etat: object
    action: int
    recompense: float                  # ACCUMULÉE sur les n pas, et actualisée
    etat_final: Optional[object]       # None SI l'épisode s'est terminé


env_jouet = JouetCompteur()
obs, _ = env_jouet.reset(seed=0)
print("trois pas du jouet :", [env_jouet.step(1)[:3] for _ in range(3)])

**D'abord, écrivez sur une feuille** les douze premières transitions que la source
d'expérience va produire avec `steps_count=2`, avec leurs **quatre champs**. Ensuite
seulement, exécutez. N'exécutez pas la cellule suivante avant d'avoir écrit vos douze
lignes : le volet perd tout son intérêt si vous lisez la réponse d'abord.

In [ ]:
def premier_dernier(trajectoire, gamma, n):
    """Replie n transitions élémentaires en une seule (section 6.3)."""
    total = 0.0
    for decalage, pas in enumerate(trajectoire[:n]):
        total += (gamma ** decalage) * pas["recompense"]
    dernier = trajectoire[n - 1]
    return TransitionPremierDernier(
        etat=trajectoire[0]["etat"],
        action=trajectoire[0]["action"],
        recompense=total,
        # Le cas terminal est encodé par l'ABSENCE d'état final.
        etat_final=None if dernier["fin"] else dernier["suivant"],
    )


...        # À COMPLÉTER : la source d'expérience, avec steps_count=2, et les douze premières transitions

**Trois questions à traiter en une ligne chacune, et voici les réponses.**

1. **Pourquoi la plupart des transitions portent-elles une récompense de 2 ?** Parce
   que l'agent joue toujours l'action 1, la récompense vaut 1 par pas, et le repliage
   à deux pas en accumule deux avec \(\gamma = 1\).
2. **Que vaut le champ d'état final à la fin d'un épisode, et pourquoi ce choix ?**
   Il vaut `None`. Un drapeau séparé peut être oublié dans une branche ; une valeur
   **absente** fait lever une erreur au premier usage, donc l'oubli se voit.
3. **Deux transitions sortent coup sur coup en fin d'épisode : laquelle est
   laquelle ?** La première est repliée sur deux pas et vaut 2 ; la seconde n'a plus
   qu'un pas devant elle et vaut 1. La seconde est plus petite parce qu'il ne reste
   **pas assez d'épisode** pour accumuler deux récompenses.

---
# Volet B : le socle commun, et la référence sur CartPole

**Objectif.** Réorganiser votre code de la semaine 5 en un **socle importable**.

Deux fichiers, et c'est le point du volet. `%%writefile` écrit une vraie cellule dans
un vrai fichier : vous pourrez ensuite lancer `python executer.py reference --graine 0`
depuis un terminal, et c'est exactement ce qu'on vous demandera au jalon.

In [ ]:
%%writefile commun.py
"""Socle commun de la semaine 6. Importé par executer.py, jamais recopié."""
import collections

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn

HYPERPARAMETRES = {
    "gamma": 0.99, "taille_lot": 64, "taille_tampon": 10_000,
    "tampon_initial": 1_000, "pas_apprentissage": 1e-3, "synchro_cible": 500,
    "epsilon_debut": 1.0, "epsilon_fin": 0.02, "epsilon_trames": 10_000,
    "seuil_resolu": 475.0,
}

Transition = collections.namedtuple(
    "Transition", ["obs", "action", "recompense", "termine", "obs_suivante"])


class Reseau(nn.Module):
    def __init__(self, n_obs, n_actions, n_cache=128):
        super().__init__()
        self.pipe = nn.Sequential(
            nn.Linear(n_obs, n_cache), nn.ReLU(),
            nn.Linear(n_cache, n_cache), nn.ReLU(),
            nn.Linear(n_cache, n_actions),
        )

    def forward(self, x):
        return self.pipe(x)


class ReseauDuel(nn.Module):
    """Deux têtes, et la soustraction de la moyenne (section 6.8).

    Largeur réduite à 120 pour que le budget de paramètres reste comparable à la
    référence : comparer un duel plus gros mesurerait la taille, pas l'architecture.
    """

    def __init__(self, n_obs, n_actions, n_cache=120):
        super().__init__()
        self.tronc = nn.Sequential(
            nn.Linear(n_obs, n_cache), nn.ReLU(),
            nn.Linear(n_cache, n_cache), nn.ReLU(),
        )
        self.tete_valeur = nn.Linear(n_cache, 1)
        self.tete_avantage = nn.Linear(n_cache, n_actions)

    def avantages(self, x):
        a = self.tete_avantage(self.tronc(x))
        return a - a.mean(dim=1, keepdim=True)

    def forward(self, x):
        h = self.tronc(x)
        a = self.tete_avantage(h)
        return self.tete_valeur(h) + a - a.mean(dim=1, keepdim=True)


def _tenseurs(lot, peripherique):
    obs, actions, recompenses, termines, suivantes = lot
    return (torch.as_tensor(obs, dtype=torch.float32, device=peripherique),
            torch.as_tensor(actions, dtype=torch.int64, device=peripherique),
            torch.as_tensor(recompenses, dtype=torch.float32, device=peripherique),
            torch.as_tensor(termines, dtype=torch.bool, device=peripherique),
            torch.as_tensor(suivantes, dtype=torch.float32, device=peripherique))


def perte_reference(lot, reseau, reseau_cible, gamma, peripherique, n=1):
    obs_t, actions_t, recompenses_t, termines_t, suiv_t = _tenseurs(lot, peripherique)
    valeurs = reseau(obs_t).gather(1, actions_t.unsqueeze(-1)).squeeze(-1)
    with torch.no_grad():
        suivantes = reseau_cible(suiv_t).max(1)[0]
        suivantes[termines_t] = 0.0
        cibles = recompenses_t + (gamma ** n) * suivantes     # gamma**n, PAS gamma
    return nn.MSELoss()(valeurs, cibles)


def perte_double(lot, reseau, reseau_cible, gamma, peripherique, n=1):
    """L'action vient du réseau ENTRAÎNÉ, son évaluation du réseau CIBLE."""
    obs_t, actions_t, recompenses_t, termines_t, suiv_t = _tenseurs(lot, peripherique)
    valeurs = reseau(obs_t).gather(1, actions_t.unsqueeze(-1)).squeeze(-1)
    with torch.no_grad():
        choix = reseau(suiv_t).argmax(dim=1)
        suivantes = reseau_cible(suiv_t).gather(1, choix.unsqueeze(-1)).squeeze(-1)
        suivantes[termines_t] = 0.0
        cibles = recompenses_t + (gamma ** n) * suivantes
    return nn.MSELoss()(valeurs, cibles)

In [ ]:
%%writefile executer.py
"""Trente lignes : choisir une variante, la lancer, enregistrer la courbe."""
import argparse
import json

import commun

# Une variante est un triplet (perte, classe de réseau, n).
VARIANTES = {
    "reference": (commun.perte_reference, commun.Reseau,     1),
    "n-etapes":  (commun.perte_reference, commun.Reseau,     3),  # seul n change
    "double":    (commun.perte_double,    commun.Reseau,     1),  # seule la perte
    "duel":      (commun.perte_reference, commun.ReseauDuel, 1),  # seul le réseau
}

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("variante", choices=sorted(VARIANTES))
    p.add_argument("--graine", type=int, required=True)   # JAMAIS de défaut implicite
    args = p.parse_args()
    print("variante", args.variante, "graine", args.graine)

### Les trois points de contrôle

In [ ]:
...        # À COMPLÉTER : le socle est importé, pas recopié

# 1. Le socle est importé, pas recopié : une seule source de vérité.
assert commun.HYPERPARAMETRES["gamma"] == 0.99

# 2. Toutes les variantes partagent la MÊME graine, passée explicitement.
graine = 0
assert graine is not None

# 3. Le réseau duel a un nombre de paramètres COMPARABLE à la référence.
reseau_reference = ...        # À COMPLÉTER : la référence
reseau_duel = ...        # À COMPLÉTER : le duel
n_ref = sum(p.numel() for p in reseau_reference.parameters())
n_duel = sum(p.numel() for p in reseau_duel.parameters())
assert 0.8 < n_duel / n_ref < 1.25        # sinon la comparaison est truquée
print(f"référence {n_ref} paramètres, duel {n_duel} ({n_duel / n_ref:.2f}x)")

---
# Volet C, évalué : l'ablation à quatre variantes

Quatre configurations, le même socle, les **mêmes trois graines**, le même lissage.

| Configuration | La seule chose qui change | Le défaut visé |
| --- | --- | --- |
| **(1) référence** | rien | la ligne de base |
| **(2) n étapes**, \(n = 3\) | `steps_count=3` et `gamma ** 3` | nº5, la cible à un pas |
| **(3) double** | trois lignes dans la perte | nº1, le \(\max\) qui surestime |
| **(4) duel** | la classe du réseau | nº3, une seule tête |

In [ ]:
GRAINES = [0, 1, 2]        # LES MÊMES pour les quatre variantes

VARIANTES = {
    "reference": dict(perte=commun.perte_reference, classe_reseau=commun.Reseau, n_etapes=1),
    "n-etapes":  dict(perte=commun.perte_reference, classe_reseau=commun.Reseau, n_etapes=3),
    "double":    dict(perte=commun.perte_double,    classe_reseau=commun.Reseau, n_etapes=1),
    "duel":      dict(perte=commun.perte_reference, classe_reseau=commun.ReseauDuel, n_etapes=1),
}

resultats = {nom: [entrainer_dqn(graine=g, max_trames=40_000, **opts) for g in GRAINES]
             for nom, opts in VARIANTES.items()}

In [ ]:
def mediane_et_extremes(courbes):
    """Médiane et extrêmes point par point, sur un axe commun.

    Les exécutions n'ont pas la même longueur : on tronque EXPLICITEMENT à la plus
    courte plutôt que d'agréger des points qui ne se correspondent pas.
    """
    n = min(len(c) for c in courbes)
    m = np.vstack([c[:n] for c in courbes])
    return np.median(m, axis=0), m.min(axis=0), m.max(axis=0)


fig, ax = plt.subplots(figsize=(9, 5))
for nom, runs in resultats.items():
    courbes = [np.convolve(r["retours"], np.ones(10) / 10, mode="valid") for r in runs]
    med, bas, haut = mediane_et_extremes(courbes)
    x = np.arange(len(med))
    ligne, = ax.plot(x, med, label=f"{nom} (n={len(runs)})")
    ax.fill_between(x, bas, haut, alpha=0.2, color=ligne.get_color(), linewidth=0)
ax.set_xlabel("épisode (lissé sur dix)"); ax.set_ylabel("retour")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [ ]:
print(f"{'configuration':<14} | {'retour médian final':>19} | {'écart min-max':>13}")
for nom, runs in resultats.items():
    finaux = [float(r["retours"][-20:].mean()) for r in runs]
    print(f"{nom:<14} | {np.median(finaux):>19.1f} | {max(finaux) - min(finaux):>13.1f}")

### Le piège à vérifier explicitement, et il vaut des points

Montrez par un **test** que la configuration (2) utilise bien \(\gamma^3\) et non
\(\gamma\). **Elle ne lève aucune exception.**

In [ ]:
class QTest(nn.Module):
    """Le même réseau que le socle, défini ici pour que la preuve soit autonome."""

    def __init__(self, n_obs, n_actions, n_cache=128):
        super().__init__()
        self.pipe = nn.Sequential(
            nn.Linear(n_obs, n_cache), nn.ReLU(),
            nn.Linear(n_cache, n_cache), nn.ReLU(),
            nn.Linear(n_cache, n_actions),
        )

    def forward(self, x):
        return self.pipe(x)


def perte_pkg(lot, reseau, reseau_cible, gamma, peripherique, n=1):
    obs, actions, recompenses, termines, suivantes = lot
    obs_t = torch.as_tensor(obs, dtype=torch.float32, device=peripherique)
    suiv_t = torch.as_tensor(suivantes, dtype=torch.float32, device=peripherique)
    actions_t = torch.as_tensor(actions, dtype=torch.int64, device=peripherique)
    recompenses_t = torch.as_tensor(recompenses, dtype=torch.float32, device=peripherique)
    termines_t = torch.as_tensor(termines, dtype=torch.bool, device=peripherique)
    valeurs = reseau(obs_t).gather(1, actions_t.unsqueeze(-1)).squeeze(-1)
    with torch.no_grad():
        suiv = reseau_cible(suiv_t).max(1)[0]
        suiv[termines_t] = 0.0
        cibles = recompenses_t + (gamma ** n) * suiv
    return nn.MSELoss()(valeurs, cibles)


# Un lot fabriqué à la main, SANS transition terminale, et un réseau figé : la
# perte à n=3 doit différer de celle à n=1. Si gamma**n était écrit gamma, les
# deux nombres seraient identiques et rien ne le signalerait.
rng = np.random.default_rng(1)
lot = (rng.normal(size=(8, 4)).astype(np.float32),
       np.zeros(8, dtype=np.int64),
       np.ones(8, dtype=np.float32),
       np.zeros(8, dtype=bool),
       rng.normal(size=(8, 4)).astype(np.float32))

torch.manual_seed(0)
reseau_t, cible_t = QTest(4, 2), QTest(4, 2)
p1 = float(perte_pkg(lot, reseau_t, cible_t, 0.5, torch.device("cpu"), n=1))
p3 = float(perte_pkg(lot, reseau_t, cible_t, 0.5, torch.device("cpu"), n=3))
print(f"perte n=1 : {p1:.6f}   perte n=3 : {p3:.6f}")
assert abs(p1 - p3) > 1e-6, "la perte ne dépend pas de n : gamma**n est absent"
print("gamma**n est bien utilisé")

**Attentes du volet C.**

* Les quatre courbes sur un seul graphique, axe en trames, **médiane et bande**.
* **Une page écrite.** Pour chacune des trois variantes : quel défaut de la section
  5.9 elle répare, par quel mécanisme, et **si l'écart mesuré est plus grand ou plus
  petit que la dispersion entre vos graines**. Mots attendus : **« surestimation »**,
  **« identifiable »**, **« dépliage »**, **« dispersion »**.
* **Une phrase que je veux lire textuellement** : ce qui, dans votre protocole,
  garantit que vous mesurez la variante et non autre chose. Ici : un socle importé
  et non recopié, les mêmes hyperparamètres, les mêmes graines, le même lissage, et
  **un seul** triplet modifié par variante.

---
# Volet D : rendre visibles deux effets invisibles

**Protocole 1, la surestimation.** Mille états témoins collectés **avant**
l'entraînement, puis la moyenne de \(\max_a Q_\theta(s,a)\) sur ces états, toutes les
mille trames, pour les configurations (1) et (3).

In [ ]:
def collecter_etats_temoins(env_id, n=1000, graine=0):
    """Mille états témoins, collectés AVANT l'entraînement (section 6.7).

    Ce sont des sondes : les mélanger au tampon reviendrait à mesurer sur les
    données d'entraînement.
    """
    env_t = gym.make(env_id)
    obs_t, _ = env_t.reset(seed=graine)
    env_t.action_space.seed(graine)
    etats = []
    while len(etats) < n:
        etats.append(np.asarray(obs_t, dtype=np.float32))
        obs_t, _, termine, tronque, _ = env_t.step(env_t.action_space.sample())
        if termine or tronque:
            obs_t, _ = env_t.reset()
    env_t.close()
    return torch.as_tensor(np.asarray(etats[:n]))


# Les états témoins ne servent JAMAIS à l'apprentissage.
etats_temoins = collecter_etats_temoins("CartPole-v1", n=1000, graine=0)
print("états témoins :", tuple(etats_temoins.shape))


@torch.no_grad()
def valeur_moyenne_temoins(reseau) -> float:
    return float(reseau(etats_temoins).max(dim=1)[0].mean())

In [ ]:
...        # À COMPLÉTER : rejouer (1) et (3) en relevant valeur_moyenne_temoins toutes les mille trames, puis tracer

**Protocole 2, la contrainte duale.** La moyenne des avantages d'un réseau duel doit
rester **proche de zéro**. Si elle dérive, votre soustraction est mal branchée.

In [ ]:
class DuelVerif(nn.Module):
    """Le réseau duel, défini ici pour que la vérification soit autonome."""

    def __init__(self, n_obs, n_actions, n_cache=120):
        super().__init__()
        self.tronc = nn.Sequential(
            nn.Linear(n_obs, n_cache), nn.ReLU(),
            nn.Linear(n_cache, n_cache), nn.ReLU(),
        )
        self.tete_valeur = nn.Linear(n_cache, 1)
        self.tete_avantage = nn.Linear(n_cache, n_actions)

    def avantages(self, x):
        a = self.tete_avantage(self.tronc(x))
        return a - a.mean(dim=1, keepdim=True)


torch.manual_seed(0)
duel = DuelVerif(4, 2)
with torch.no_grad():
    avantages = duel.avantages(torch.randn(256, 4))
print(f"moyenne des avantages : {float(avantages.mean()):.2e}")
print(f"amplitude maximale par ligne : {float(avantages.mean(dim=1).abs().max()):.2e}")
assert avantages.mean(dim=1).abs().max() < 1e-5

**Attentes du volet D.**

* Deux graphiques, et **trois phrases** sur chacun.
* Sur le premier : la version ordinaire s'attribue-t-elle des valeurs qu'elle finit
  par démentir ? Et **pouvez-vous le conclure sur une seule graine** ? Non : il faut
  au moins trois graines, parce que la dispersion entre exécutions est du même ordre
  que l'écart entre les deux courbes.
* Sur le second : la moyenne des avantages reste au niveau du bruit numérique,
  autour de \(10^{-8}\).
* **Pourquoi ces deux mesures sont-elles indispensables, alors que la courbe de
  retour ne montre ni l'une ni l'autre ?** Parce que la courbe de retour mesure le
  **résultat** et ces deux-là mesurent le **mécanisme** : une surestimation peut
  croître longtemps avant que le retour ne s'en ressente, et une soustraction mal
  branchée peut laisser le retour intact tout en rendant les deux têtes
  ininterprétables.

---
# Volet E, facultatif : les deux réparations les plus intrusives

**Piste 1, les réseaux bruités.** Remplacez les deux couches linéaires de la tête, et
**supprimez** tout le code de la stratégie \(\varepsilon\)-gloutonne.

In [ ]:
class CoucheBruitee(nn.Module):
    """Couche linéaire à bruit factorisé (section 6.9)."""

    def __init__(self, n_entree, n_sortie, sigma_initial=0.5):
        super().__init__()
        self.n_entree, self.n_sortie = n_entree, n_sortie
        borne = 1.0 / math.sqrt(n_entree)
        self.mu_poids = nn.Parameter(torch.empty(n_sortie, n_entree).uniform_(-borne, borne))
        self.sigma_poids = nn.Parameter(
            torch.full((n_sortie, n_entree), sigma_initial / math.sqrt(n_entree)))
        self.mu_biais = nn.Parameter(torch.empty(n_sortie).uniform_(-borne, borne))
        self.sigma_biais = nn.Parameter(
            torch.full((n_sortie,), sigma_initial / math.sqrt(n_entree)))
        self.register_buffer("eps_entree", torch.zeros(n_entree))
        self.register_buffer("eps_sortie", torch.zeros(n_sortie))
        self.retirer_bruit()

    @staticmethod
    def _echelle(x):
        return x.sign() * x.abs().sqrt()

    def retirer_bruit(self):
        """Tirage EXPLICITE : c'est lui dont la semaine 11 fait varier la fréquence."""
        self.eps_entree.copy_(self._echelle(torch.randn(self.n_entree)))
        self.eps_sortie.copy_(self._echelle(torch.randn(self.n_sortie)))

    @torch.no_grad()
    def rapport_signal_bruit(self):
        return float(self.mu_poids.norm() / (self.sigma_poids.norm() + 1e-12))

    def forward(self, x):
        if not self.training:
            return nn.functional.linear(x, self.mu_poids, self.mu_biais)
        poids = self.mu_poids + self.sigma_poids * self.eps_sortie.outer(self.eps_entree)
        biais = self.mu_biais + self.sigma_biais * self.eps_sortie
        return nn.functional.linear(x, poids, biais)


# La vérification demandée : eval() rend l'agent déterministe.
torch.manual_seed(0)
couche = CoucheBruitee(4, 3)
x = torch.zeros(1, 4)
couche.train()
a = couche(x); couche.retirer_bruit(); b = couche(x)
couche.eval()
c, d = couche(x), couche(x)
print("en entraînement, après un nouveau tirage :", not torch.allclose(a, b))
print("en évaluation, deux passes identiques   :", torch.allclose(c, d))
print(f"rapport signal/bruit initial : {couche.rapport_signal_bruit():.2f}")

**Piste 2, le tampon priorisé.** Le **seul** réglage qui compte : vérifier qu'à
\(\alpha_{\mathrm{PER}} = 0\) le tampon reproduit **exactement** le tirage uniforme.
Si ce n'est pas le cas, votre tampon a un bogue, et ce test est le plus rapide pour
le trouver.

In [ ]:
pri = TamponPriorise(100, alpha=0.0, graine=3)
uni = TamponUniforme(100, graine=3)
for i in range(40):
    t = T(np.full(3, float(i), dtype=np.float32), i % 2, float(i), False,
          np.full(3, i + 1.0, dtype=np.float32))
    pri.ajouter(t); uni.ajouter(t)
pri.mettre_a_jour(np.arange(40), np.random.default_rng(0).random(40) * 10)

*_, r_pri, _, _, _, _ = (*pri.echantillonner(10),)
r_uni = uni.echantillonner(10)[2]
print("alpha=0 reproduit le tirage uniforme :",
      np.array_equal(np.sort(pri.echantillonner(10)[2]), np.sort(r_uni)))

**Critères de réussite du volet E.**

* Piste 1 : la courbe de retour, la courbe du rapport signal sur bruit, et une phrase
  sur la question qui compte, à savoir **si votre rapport remonte quelque part**.
* Piste 2 : la vérification \(\alpha_{\mathrm{PER}} = 0\), et une mesure en **deux**
  unités, pas d'apprentissage et temps écoulé. Si vos deux unités donnent deux
  conclusions opposées, vous venez de reproduire le résultat de la section 6.10.

> **Mon message pour ce laboratoire.** Vous allez trouver, je le crois, que sur
> CartPole les écarts entre variantes sont du même ordre que l'écart entre deux
> graines de la même variante. Ce serait un **excellent** résultat, et il ne faut
> surtout pas le cacher. **Un résultat nul correctement établi vaut mieux qu'un écart
> impressionnant obtenu sur une seule exécution.**